In [1]:
!pip install ultralytics
!pip install roboflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.0/285.0 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 133.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 6.8 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 5.0.0.93
    Uninstalling opencv-python-headless-5.0.0.93:
      Successfully uninstalled opencv-python-headless-5.0.0.93
  Attempting uninstall: typer
    Found existing installation: typer 0.26.8
    Uninstalling typer-0.26.8:
      Successfully uninstalled typer-0.2

In [13]:
from roboflow import Roboflow

rf = Roboflow(api_key="0oTc4c7hPC7bb1dx3vEL")

project = rf.workspace("nevas-workspace-0qzfn").project("findric")
version = project.version(6)

dataset = version.download("yolov8")

print(f"hi, {dataset}")
print(f"{dataset.location}")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to FINDRIC-6 in yolov8:: 100%|██████████| 993/993 [00:00<00:00, 7013.66it/s]

hi, <roboflow.core.dataset.Dataset object at 0x7a9c9a172c60>
/content/FINDRIC-6


In [14]:
import yaml
import os
from ultralytics import YOLO

# 1. 定義路徑 (根據你之前的報錯資訊)
dataset_root = '/content/FINDRIC-6'
yaml_path = f"{dataset_root}/data.yaml"

# 2. 自動修正 data.yaml 的路徑問題
if os.path.exists(yaml_path):
    with open(yaml_path, 'r') as f:
        config = yaml.safe_load(f)

    # 修正為 Colab 的絕對路徑
    config['path'] = dataset_root
    config['train'] = 'train/images'
    config['val'] = 'train/images'  # 應急方案：若無 valid 資料夾，先用 train 跑通訓練 [cite: 109, 130]

    # 寫回修正後的設定
    with open(yaml_path, 'w') as f:
        yaml.dump(config, f)
    print("✅ data.yaml 路徑已修正完成！")
else:
    print(f"❌ 找不到 YAML 檔案，請確認路徑是否為: {yaml_path}")

# 3. 確保訓練環境 (應對可能的資料夾缺失)
!mkdir -p {dataset_root}/valid/images

# 4. 加載模型並開始「過擬合」訓練
model = YOLO('yolov8n.pt')

# 開始訓練 (已修正 data 參數指向)
results = model.train(
    data=yaml_path,              # 使用剛才修正過的絕對路徑 [cite: 91, 109]
    epochs=100,                  # 資料量少，100次內會收斂 [cite: 109, 130]
    imgsz=640,                   # 圖片尺寸標準化 [cite: 188]
    plots=True,                  # 顯示訓練圖表 (報告必備) [cite: 101, 105]
    device=0,                    # 使用 T4 GPU (請確保已切換執行階段) [cite: 159, 160]
    save=True,                   # 儲存模型權重 [cite: 109]
    name='RIC_HappyCase_v2'      # 為這次實驗命名，方便查找結果
)

✅ data.yaml 路徑已修正完成！
Ultralytics 8.4.119 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/FINDRIC-6/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=RIC_HappyC

In [17]:
# 載入最強的權重
import cv2
from ultralytics import YOLO
from IPython.display import Image

best_model = YOLO('/content/runs/detect/RIC_HappyCase_v2-2/weights/best.pt')

# 請放一張你沒有放進 Roboflow 訓練的照片路徑
# test_image = "IMG_8194.JPG"

# 執行識別
# results = best_model.predict(source=test_image, conf=0.1, show=True, save=True)

# 顯示結果圖
# 注意：預測結果會存在 runs/detect/predict/ 資料夾下
# Image(filename='/content/runs/detect/predict/IMG_8234.JPG')

# 核心：識別前必須先進行預處理！
def get_ready_image(path):
    img = cv2.imread(path, 0)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(img)
    edges = cv2.Canny(enhanced, 40, 120)
    edges_3ch = cv2.cvtColor(edges, cv2.COLOR_GRAY2RGB)
    return edges_3ch

# 先轉化為邊緣圖 [cite: 358]
processed_img = get_ready_image('/content/testdata/pet1test_2.JPG')

# 進行預測，並嘗試降低 conf
results = best_model.predict(source=processed_img, conf=0.2, save=True, show=False)



0: 640x480 1 pp5, 6.1ms
Speed: 4.0ms preprocess, 6.1ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 480)
Results saved to /content/runs/detect/predict-4


In [18]:
# 1. Zip the entire 'runs' folder
!zip -r /content/runs.zip /content/runs

# 2. Download the zipped file to your computer
from google.colab import files
files.download('/content/runs.zip')



updating: content/runs/ (stored 0%)
updating: content/runs/detect/ (stored 0%)
updating: content/runs/detect/predict/ (stored 0%)
updating: content/runs/detect/predict/image0.jpg (deflated 4%)
updating: content/runs/detect/RIC_HappyCase_v2/ (stored 0%)
updating: content/runs/detect/RIC_HappyCase_v2/results.png (deflated 8%)
updating: content/runs/detect/RIC_HappyCase_v2/BoxR_curve.png (deflated 17%)
updating: content/runs/detect/RIC_HappyCase_v2/results.csv (deflated 61%)
updating: content/runs/detect/RIC_HappyCase_v2/BoxF1_curve.png (deflated 16%)
updating: content/runs/detect/RIC_HappyCase_v2/val_batch1_pred.jpg (deflated 25%)
updating: content/runs/detect/RIC_HappyCase_v2/val_batch1_labels.jpg (deflated 25%)
updating: content/runs/detect/RIC_HappyCase_v2/train_batch2.jpg (deflated 16%)
updating: content/runs/detect/RIC_HappyCase_v2/train_batch1.jpg (deflated 16%)
updating: content/runs/detect/RIC_HappyCase_v2/train_batch632.jpg (deflated 19%)
updating: content/runs/detect/RIC_HappyC

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>